# Smart Agriculture Analytics - Complete Solution

This notebook contains the complete solution for all questions (Q1-Q7) from the case study.

**Run each cell in order** to see the output step by step.

## Step 0: Import Libraries

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Q1. Spark Initialization & Data Loading (2 Marks)

In [ ]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("SmartAgricultureAnalytics") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

sc = spark.sparkContext

print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
# Load all datasets
weather_df = spark.read.csv("data/weather_data.csv", header=True, inferSchema=True)
soil_df = spark.read.csv("data/soil_data.csv", header=True, inferSchema=True)
crop_df = spark.read.csv("data/crop_data.csv", header=True, inferSchema=True)
iot_df = spark.read.csv("data/iot_sensor_data.csv", header=True, inferSchema=True)

print("\nWEATHER DATA:")
weather_df.printSchema()
weather_df.show(5)

print("\nSOIL DATA:")
soil_df.printSchema()
soil_df.show(5)

print("\nCROP DATA:")
crop_df.printSchema()
crop_df.show(5)

print("\nIOT DATA:")
iot_df.printSchema()
iot_df.show(5)

## Q2. RDD Implementation (3 Marks)

In [ ]:
# Convert DataFrames to RDDs
weather_rdd = weather_df.rdd
soil_rdd = soil_df.rdd
crop_rdd = crop_df.rdd
iot_rdd = iot_df.rdd

print("TRANSFORMATIONS:")

# 1. map()
print("\n1. map() - Celsius to Fahrenheit:")
temp_f = weather_rdd.map(lambda row: (row.station_id, row.temperature * 9/5 + 32))
for item in temp_f.take(5):
    print(f"  {item}")

# 2. filter()
print("\n2. filter() - High rainfall > 150mm:")
high_rain = weather_rdd.filter(lambda row: row.rainfall > 150)
print(f"  Count: {high_rain.count()}")

# 3. flatMap()
print("\n3. flatMap() - Region characters:")
chars = weather_rdd.flatMap(lambda row: list(row.region)).take(10)
print(f"  {chars}")

# 4. union()
print("\n4. union() - Combine temps:")
combined = weather_rdd.map(lambda r: ("weather", r.temperature)).union(
    iot_rdd.map(lambda r: ("iot", r.temperature))
)
print(f"  Total: {combined.count()}")

In [ ]:
print("ACTIONS:")

# 1. collect()
print("\n1. collect() - All regions:")
regions = weather_rdd.map(lambda r: r.region).collect()
print(f"  {set(regions)}")

# 2. count()
print(f"\n2. count() - Total records: {weather_rdd.count()}")

# 3. reduce()
print("\n3. reduce() - Total rainfall:")
total = weather_rdd.map(lambda r: r.rainfall).reduce(lambda a, b: a + b)
print(f"  {total:.2f} mm")

# 4. first()
print(f"\n4. first() - {weather_rdd.first()}")

# 5. take()
print("\n5. take() - Top 3 temps:")
top3 = weather_rdd.takeOrdered(3, key=lambda r: -r.temperature)
for r in top3:
    print(f"  {r.station_id}: {r.temperature}C")

# 6. distinct()
print(f"\n6. distinct() - Soil types: {soil_rdd.map(lambda r: r.soil_type).distinct().collect()}")

## Q3. Key-Value Operations & Persistence (2 Marks)

In [ ]:
# Create Key-Value RDDs
weather_kv = weather_rdd.map(lambda r: (r.region, r.temperature))
crop_kv = crop_rdd.map(lambda r: (r.region, r.yield_tons))

print("Key-Value samples:")
print(weather_kv.take(5))
print(crop_kv.take(5))

# 1. reduceByKey()
print("\nreduceByKey() - Avg temp by region:")
avg_temp = weather_kv.mapValues(lambda x: (x, 1)) \
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
    .mapValues(lambda x: x[0]/x[1])
for r, t in avg_temp.collect():
    print(f"  {r}: {t:.2f}C")

# 2. groupByKey()
print("\ngroupByKey() - Temps by region:")
grouped = weather_kv.groupByKey()
for r, temps in grouped.collect():
    print(f"  {r}: {list(temps)[:3]}...")

# 3. join()
print("\njoin() - Weather + Crop:")
crop_avg = crop_kv.mapValues(lambda x: (x, 1)) \
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
    .mapValues(lambda x: x[0]/x[1])
joined = avg_temp.join(crop_avg)
for r, (t, y) in joined.take(5):
    print(f"  {r}: Temp={t:.2f}C, Yield={y:.2f} tons")

# 4. sortByKey()
print("\nsortByKey():")
for r, t in avg_temp.sortByKey().collect():
    print(f"  {r}: {t:.2f}C")

In [ ]:
# Persistence
from pyspark import StorageLevel

print("PERSISTENCE:")

weather_kv.persist()
print(f"1. persist(): {weather_kv.getStorageLevel()}")
weather_kv.unpersist()
print("2. unpersist(): Removed")

weather_kv.persist(StorageLevel.MEMORY_ONLY)
print("3. MEMORY_ONLY: Stored in RAM")
weather_kv.unpersist()

weather_kv.persist(StorageLevel.MEMORY_AND_DISK)
print("4. MEMORY_AND_DISK: RAM + Disk")
weather_kv.unpersist()

weather_kv.persist(StorageLevel.DISK_ONLY)
print("5. DISK_ONLY: Disk only")
weather_kv.unpersist()

print("\nPersistence demo complete!")

## Q4. Spark DataFrame Operations (3 Marks)

In [ ]:
print("SELECTION:")
weather_df.select("station_id", "region", "temperature").show(5)

weather_df.selectExpr("station_id", "temperature * 9/5 + 32 as temp_f").show(5)

print("\nFILTERING:")
weather_df.filter(col("temperature") > 35).show(5)
weather_df.filter((col("region") == "North") & (col("rainfall") > 100)).show(5)

print("\nGROUPING & AGGREGATION:")
weather_df.groupBy("region").agg(
    avg("rainfall").alias("avg_rain"),
    max("temperature").alias("max_temp"),
    count("*").alias("count")
).show()

crop_df.groupBy("crop_name").agg(
    avg("yield_tons").alias("avg_yield"),
    sum("yield_tons").alias("total_yield")
).orderBy(desc("avg_yield")).show()

In [ ]:
print("JOINS:")
weather_df.join(crop_df, weather_df.region == crop_df.region, "inner").select(
    "station_id", weather_df.region, "temperature", "crop_name", "yield_tons"
).show(10)

print("\nADVANCED:")
crop_df.withColumn("yield_per_hectare", round(col("yield_tons")/col("area_hectares"), 2)).show(5)
crop_df.drop("irrigation_type").printSchema()
print(f"Unique regions: {crop_df.select('region').distinct().count()}")
crop_df.orderBy(desc("yield_tons")).show(5)
weather_df.limit(3).show()
weather_df.describe().show()

## Q5. EDA & Spark SQL (5 Marks)

In [ ]:
# Register temporary views
weather_df.createOrReplaceTempView("weather")
soil_df.createOrReplaceTempView("soil")
crop_df.createOrReplaceTempView("crops")
iot_df.createOrReplaceTempView("iot_sensors")

print("1. CROP-WISE PRODUCTION TRENDS:")
spark.sql("""
    SELECT crop_name, year, season,
           ROUND(AVG(yield_tons), 2) as avg_yield,
           ROUND(SUM(yield_tons), 2) as total_yield
    FROM crops
    GROUP BY crop_name, year, season
    ORDER BY crop_name, year, season
""").show(20)

print("\n2. RAINFALL & TEMPERATURE PATTERNS:")
spark.sql("""
    SELECT region,
           ROUND(AVG(rainfall), 2) as avg_rainfall,
           ROUND(AVG(temperature), 2) as avg_temp,
           ROUND(MAX(temperature), 2) as max_temp,
           ROUND(MIN(temperature), 2) as min_temp
    FROM weather
    GROUP BY region
    ORDER BY avg_rainfall DESC
""").show()

print("\n3. REGION-WISE CROP YIELDS:")
spark.sql("""
    SELECT c.region, c.crop_name,
           ROUND(AVG(c.yield_tons), 2) as avg_yield,
           ROUND(SUM(c.yield_tons), 2) as total_yield
    FROM crops c
    GROUP BY c.region, c.crop_name
    ORDER BY c.region, avg_yield DESC
""").show(20)

print("\n4. TOP PERFORMING REGIONS:")
spark.sql("""
    SELECT region,
           ROUND(SUM(yield_tons), 2) as total_yield,
           ROUND(AVG(yield_tons), 2) as avg_yield,
           COUNT(DISTINCT crop_name) as crop_variety
    FROM crops
    GROUP BY region
    ORDER BY total_yield DESC
""").show()

print("\n5. SEASONAL CROP PRODUCTION:")
spark.sql("""
    SELECT season, crop_name,
           ROUND(AVG(yield_tons), 2) as avg_yield,
           ROUND(SUM(yield_tons), 2) as total_yield
    FROM crops
    GROUP BY season, crop_name
    ORDER BY season, total_yield DESC
""").show(20)

## Q6. ETL Pipeline (3 Marks)

In [ ]:
print("ETL PIPELINE: Extract -> Transform -> Load\n" + "-"*50)

# EXTRACT
print("\nSTEP 1: EXTRACT")
raw_iot = spark.read.csv("data/iot_sensor_data.csv", header=True, inferSchema=True)
raw_weather = spark.read.csv("data/weather_data.csv", header=True, inferSchema=True)
raw_soil = spark.read.csv("data/soil_data.csv", header=True, inferSchema=True)
raw_crops = spark.read.csv("data/crop_data.csv", header=True, inferSchema=True)
print(f"  Extracted: IoT={raw_iot.count()}, Weather={raw_weather.count()}, Soil={raw_soil.count()}, Crops={raw_crops.count()}")

# TRANSFORM
print("\nSTEP 2: TRANSFORM")
clean_iot = raw_iot.dropna(subset=["soil_moisture", "temperature"])
clean_weather = raw_weather.withColumn("region", upper(trim(col("region"))))
clean_soil = raw_soil.withColumn("region", upper(trim(col("region"))))
clean_crops = raw_crops.withColumn("region", upper(trim(col("region"))))

enriched = clean_crops \
    .withColumn("yield_per_hectare", round(col("yield_tons")/col("area_hectares"), 2))

agg_iot = clean_iot.groupBy("region", to_date("timestamp").alias("date")).agg(
    avg("soil_moisture").alias("avg_moisture"),
    avg("temperature").alias("avg_temp")
)

etl_data = enriched.join(
    clean_weather.select("region", "temperature", "rainfall", "humidity"),
    "region", "left"
).join(
    clean_soil.select("region", "ph_level", "soil_type", "nitrogen", "phosphorus", "potassium"),
    "region", "left"
)

print("  Transform complete!")
etl_data.printSchema()

# LOAD
print("\nSTEP 3: LOAD")
etl_data.write.mode("overwrite").parquet("output/centralized_agriculture_data")
etl_data.write.mode("overwrite").option("header", "true").csv("output/centralized_agriculture_csv")
agg_iot.write.mode("overwrite").parquet("output/aggregated_iot_data")
print("  Data loaded to output/ folder!")

## Q7. Machine Learning (2 Marks)

In [ ]:
print("ML PIPELINE: Crop Yield Prediction\n" + "-"*50)

# Prepare data
ml_data = etl_data.select(
    "crop_name", "region", "season",
    "area_hectares", "fertilizer_used",
    "temperature", "rainfall", "humidity",
    "ph_level", "nitrogen", "phosphorus", "potassium",
    "yield_tons"
).dropna()

print(f"ML Dataset: {ml_data.count()} records")

# Feature Engineering Pipeline
crop_idx = StringIndexer(inputCol="crop_name", outputCol="crop_index")
region_idx = StringIndexer(inputCol="region", outputCol="region_index")
season_idx = StringIndexer(inputCol="season", outputCol="season_index")

crop_enc = OneHotEncoder(inputCol="crop_index", outputCol="crop_enc")
region_enc = OneHotEncoder(inputCol="region_index", outputCol="region_enc")
season_enc = OneHotEncoder(inputCol="season_index", outputCol="season_enc")

assembler = VectorAssembler(
    inputCols=["crop_enc", "region_enc", "season_enc",
               "area_hectares", "fertilizer_used", "temperature",
               "rainfall", "humidity", "ph_level",
               "nitrogen", "phosphorus", "potassium"],
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="yield_tons",
    predictionCol="predicted_yield",
    maxIter=50,
    regParam=0.1
)

pipeline = Pipeline(stages=[
    crop_idx, region_idx, season_idx,
    crop_enc, region_enc, season_enc,
    assembler, lr
])

# Split and train
train, test = ml_data.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()}, Test: {test.count()}")

model = pipeline.fit(train)
predictions = model.transform(test)

print("\nPredictions:")
predictions.select("crop_name", "region", "yield_tons", "predicted_yield").show(10)

In [ ]:
# Evaluation
print("MODEL EVALUATION:")
print("-" * 50)

rmse = RegressionEvaluator(labelCol="yield_tons", predictionCol="predicted_yield", metricName="rmse").evaluate(predictions)
mae = RegressionEvaluator(labelCol="yield_tons", predictionCol="predicted_yield", metricName="mae").evaluate(predictions)
r2 = RegressionEvaluator(labelCol="yield_tons", predictionCol="predicted_yield", metricName="r2").evaluate(predictions)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

# Save report
report = f"""
MODEL EVALUATION REPORT
======================
RMSE: {rmse:.4f}
MAE: {mae:.4f}
R2: {r2:.4f}
"""
with open("output/model_evaluation_report.txt", "w") as f:
    f.write(report)

# Save model
model.write().overwrite().save("output/crop_yield_model")
print("\nModel and report saved!")

## Visualization

In [ ]:
weather_pd = weather_df.toPandas()
crop_pd = crop_df.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Smart Agriculture Dashboard', fontsize=16)

# Chart 1
ax1 = axes[0, 0]
crop_yield = crop_pd.groupby('crop_name')['yield_tons'].mean().sort_values()
crop_yield.plot(kind='barh', ax=ax1, color='green', alpha=0.7)
ax1.set_title('Avg Yield by Crop')

# Chart 2
ax2 = axes[0, 1]
weather_pd.boxplot(column='temperature', by='region', ax=ax2)
ax2.set_title('Temp by Region')

# Chart 3
ax3 = axes[1, 0]
ax3.scatter(weather_pd['temperature'], weather_pd['rainfall'], alpha=0.6)
ax3.set_title('Rainfall vs Temp')

# Chart 4
ax4 = axes[1, 1]
season_yield = crop_pd.groupby('season')['yield_tons'].sum()
ax4.pie(season_yield, labels=season_yield.index, autopct='%1.1f%%')
ax4.set_title('Yield by Season')

plt.tight_layout()
plt.savefig('output/agriculture_dashboard.png', dpi=150)
plt.show()
print("Dashboard saved!")

## Save EDA Outputs

In [ ]:
# Save all EDA queries
q1 = spark.sql("SELECT crop_name, year, season, ROUND(AVG(yield_tons),2) as avg_yield FROM crops GROUP BY crop_name, year, season")
q2 = spark.sql("SELECT region, ROUND(AVG(rainfall),2) as avg_rainfall, ROUND(AVG(temperature),2) as avg_temp FROM weather GROUP BY region")
q3 = spark.sql("SELECT region, crop_name, ROUND(AVG(yield_tons),2) as avg_yield FROM crops GROUP BY region, crop_name")
q4 = spark.sql("SELECT region, ROUND(SUM(yield_tons),2) as total_yield FROM crops GROUP BY region ORDER BY total_yield DESC")
q5 = spark.sql("SELECT season, crop_name, ROUND(SUM(yield_tons),2) as total_yield FROM crops GROUP BY season, crop_name")

q1.write.mode("overwrite").csv("output/eda_crop_trends", header=True)
q2.write.mode("overwrite").csv("output/eda_weather_patterns", header=True)
q3.write.mode("overwrite").csv("output/eda_region_yields", header=True)
q4.write.mode("overwrite").csv("output/eda_top_regions", header=True)
q5.write.mode("overwrite").csv("output/eda_seasonal_production", header=True)

print("All EDA outputs saved!")
spark.stop()
print("\nSpark stopped. All done!")